# TfL Tube Strikes → Cycle Hire Usage (Data Pipeline)

This notebook builds a modelling-ready dataset for a causal ML / meta-learner analysis of:

> **How Tube strikes change cycle hire demand, and how that effect varies by time and location.**

We do four things in order:

1. **Aggregate raw journey extracts** into a station–time panel (station-hour by default).
2. **Load bike station locations** (BikePoint API dump).
3. **Fetch Tube stations + served Underground lines** (TfL StopPoint API) and build a **bike-station → Underground-line** exposure map.
4. **Expand strike events** (line-level, date ranges) and **join strike exposure** onto the station–time panel.

Outputs:
- `base_station_time.parquet` (or CSV): station_id × ts × trips (+ features you add later)
- `station_line_map.csv`: station_id × affected_line (plus nearest tube station and distance)


In [ ]:
# Imports + project utilities
import pandas as pd

# Make sure tfl_utils.py is on your Python path.
# If it's in the same folder as this notebook, this will work.
from tfl_utils import (
    aggregate_from_folder_chunked,
    load_bike_station_locations,
    fetch_tube_stations_and_lines,
    build_station_line_map,
    expand_strikes_daily,
    attach_strikes_to_base,
)

pd.set_option("display.max_columns", 50)


## 0) Configure paths

Edit these for your machine.

- `JOURNEY_FOLDER` should contain all the raw TfL journey CSV extracts you downloaded.
- `STRIKES_CSV` is your line-level strike table (`date_start`, `date_end`, `affected_line`).
- `BIKEPOINTS_JSON` is the JSON dump from the BikePoint API (all docking stations).


In [ ]:
from pathlib import Path

# Raw journey extracts (many CSVs)
JOURNEY_FOLDER = Path(r"E:\tfl_project\data")

# Your strike events table
STRIKES_CSV = Path(r"E:\tfl_project\strikes.csv")

# BikePoint station locations (downloaded JSON)
BIKEPOINTS_JSON = Path(r"E:\tfl_project\bikepoints.json")

# Where to save outputs
OUT_DIR = Path(r"E:\tfl_project\outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)


## 1) Build the station–time outcome panel

We aggregate the journey extracts into either:
- **station-hour** (recommended), or
- station-day (if you want a coarser unit).

`trips_start` counts journeys starting at station_id during that hour/day.

Notes:
- We read in chunks to avoid memory crashes.
- The extract schema changes across years; the utility detects the right columns per file.


In [ ]:
base = aggregate_from_folder_chunked(
    JOURNEY_FOLDER,
    freq="h",         # "h" for hourly, "d" for daily
    side="start",     # "start" is usually the cleanest "demand" measure
    chunksize=150_000 # adjust down if you run out of RAM
)

base.head()

In [ ]:
print("Rows:", len(base))
print("Unique stations:", base["station_id"].nunique())
print("Time range:", base["ts"].min(), "→", base["ts"].max())


## 2) Load bike station locations

We need bike station `lat/lon` to map each bike station to nearby Tube stations.

The BikePoint JSON contains all docking stations with:
- `id` like `BikePoints_123`
- `lat`, `lon`
- `commonName`


In [ ]:
bike_stations = load_bike_station_locations(BIKEPOINTS_JSON)
bike_stations.head()

## 3) Fetch Tube stations + served Underground lines

We pull Tube stations from TfL StopPoint API and keep only true Underground lines.

This produces:
- `tube_stations`: station ids + lat/lon
- `tube_lines`: mapping from tube_station_id → affected_line (e.g. `central_line`)


In [ ]:
tube_stations, tube_lines = fetch_tube_stations_and_lines()
tube_stations.head(), tube_lines.head()

## 4) Build station → line exposure mapping

Definition (simple and defensible):
- A bike station is "connected" to an Underground line if it is within `radius_m` of **any** Tube station served by that line.
- If a bike station has no Tube station within radius, we can optionally map it to the nearest Tube station (`fallback_to_nearest=True`).

We then **deduplicate** to one row per `(station_id, affected_line)` using the nearest Tube station record.


In [ ]:
station_line_map = build_station_line_map(
    bike_stations=bike_stations,
    tube_stations=tube_stations,
    tube_lines=tube_lines,
    radius_m=800,              # walking catchment radius
    fallback_to_nearest=True,
    dedupe_line_per_station=True,
)

station_line_map.head()

In [ ]:
# Sanity checks: number of lines per bike station
desc = station_line_map.groupby("station_id")["affected_line"].nunique().describe()
desc

## 5) Load strike events and expand to daily

Your strike table is line-level, with date ranges:
- `date_start`, `date_end`, `affected_line`

We expand into:
- one row per **date × affected_line**.


In [ ]:
strike_data = pd.read_csv(STRIKES_CSV)
strike_data.head()

In [ ]:
strikes_daily = expand_strikes_daily(strike_data)
strikes_daily.head()

## 6) Attach strike exposure to the station–time panel

We define binary exposure:

> `strike_exposed = 1` for station `s` on day `d` if **any** line connected to station `s`
> is on strike on day `d`.

This yields a station-hour panel with an added `strike_exposed` indicator.


In [ ]:
base_with_strikes = attach_strikes_to_base(
    base=base,
    strikes_daily=strikes_daily,
    station_line_map=station_line_map,
)

base_with_strikes.head()

In [ ]:
# Treatment prevalence sanity check
print("Share of station-hours treated:", base_with_strikes["strike_exposed"].mean())

daily_flag = (
    base_with_strikes.assign(date=base_with_strikes["ts"].dt.floor("D"))
    .groupby("date")["strike_exposed"]
    .max()
)
print("Number of strike days flagged in panel:", int(daily_flag.sum()))
daily_flag.value_counts()


## 7) Save outputs

For modelling, prefer Parquet (fast + compact).  
If you don't have `pyarrow` installed in your environment, install it in your `.venv`:

```powershell
python -m pip install pyarrow
```


In [ ]:
# Save station_line_map (small, useful to inspect)
station_line_map.to_csv(OUT_DIR / "station_line_map.csv", index=False)

# Save modelling basefile
try:
    base_with_strikes.to_parquet(OUT_DIR / "base_station_hour.parquet", index=False)
    print("Saved:", OUT_DIR / "base_station_hour.parquet")
except Exception as e:
    print("Parquet save failed (pyarrow missing?). Falling back to CSV. Error:", e)
    base_with_strikes.to_csv(OUT_DIR / "base_station_hour.csv", index=False)
    print("Saved:", OUT_DIR / "base_station_hour.csv")


## 8) Quick visual sanity checks (optional)

These are *descriptive* checks (not causal estimates).

- Total trips over time
- Mark strike days


In [ ]:
import matplotlib.pyplot as plt

daily_total = (
    base_with_strikes.assign(date=base_with_strikes["ts"].dt.floor("D"))
    .groupby("date", as_index=False)["trips_start"]
    .sum()
    .sort_values("date")
)

daily_strike = (
    base_with_strikes.assign(date=base_with_strikes["ts"].dt.floor("D"))
    .groupby("date", as_index=False)["strike_exposed"]
    .max()
)

daily = daily_total.merge(daily_strike, on="date", how="left").fillna({"strike_exposed": 0})

plt.figure()
plt.plot(daily["date"], daily["trips_start"])
plt.scatter(daily.loc[daily["strike_exposed"] == 1, "date"],
            daily.loc[daily["strike_exposed"] == 1, "trips_start"],
            marker="x")
plt.title("Total daily cycle hire starts (strike days marked)")
plt.xlabel("Date")
plt.ylabel("Trips (starts)")
plt.xticks(rotation=45)
plt.tight_layout()
plt.show()
